In [77]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

In [78]:
site_features = gpd.read_parquet(
    "../../data/processed/site_features/property_site_features_zoning_heritage_bushfire_flood_sample.parquet"
)

stations_gdf = gpd.read_parquet(
    "../../data/processed/transport/rail_metro_stations_raw.parquet"
)

In [79]:
print("Site rows:", len(site_features))
print("Station rows:", len(stations_gdf))

print("Site CRS:", site_features.crs)
print("Station CRS:", stations_gdf.crs)

print(site_features.columns)
print(stations_gdf.columns)

Site rows: 49950
Station rows: 1284
Site CRS: None
Station CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon"

In [80]:
if not isinstance(site_features, gpd.GeoDataFrame):
    site_features = gpd.GeoDataFrame(site_features, geometry="geometry")

if site_features.crs is None:
    site_features = site_features.set_crs("EPSG:4326")

if not isinstance(stations_gdf, gpd.GeoDataFrame):
    stations_gdf = gpd.GeoDataFrame(stations_gdf, geometry="geometry")

if stations_gdf.crs is None:
    stations_gdf = stations_gdf.set_crs("EPSG:4326")

print("Fixed Site CRS:", site_features.crs)
print("Fixed Station CRS:", stations_gdf.crs)

Fixed Site CRS: EPSG:4326
Fixed Station CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "eas

In [81]:
stations_gdf.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code,geometry
0,2000171,NaN,QVB,-33.871692,151.206656,1.0,<NA>,0.0,<NA>,<NA>,POINT (151.20666 -33.87169)
1,200060,NaN,Central Station,-33.884024,151.206203,1.0,<NA>,0.0,<NA>,<NA>,POINT (151.2062 -33.88402)
2,201510,NaN,Redfern Station,-33.892048,151.198419,1.0,<NA>,0.0,<NA>,<NA>,POINT (151.19842 -33.89205)
3,204210,NaN,Newtown Station,-33.897913,151.179377,1.0,<NA>,0.0,<NA>,<NA>,POINT (151.17938 -33.89791)
4,204420,NaN,Sydenham Station,-33.914459,151.166526,1.0,<NA>,0.0,<NA>,<NA>,POINT (151.16653 -33.91446)


In [82]:
if "location_type" in stations_gdf.columns:
    print(stations_gdf["location_type"].value_counts(dropna=False))
else:
    print("No location_type column found.")

location_type
1.0    1284
Name: count, dtype: int64


In [83]:
station_points = stations_gdf.copy()
print("Station points kept:", len(station_points))
station_points.head()

Station points kept: 1284


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code,geometry
0,2000171,NaN,QVB,-33.871692,151.206656,1.0,<NA>,0.0,<NA>,<NA>,POINT (151.20666 -33.87169)
1,200060,NaN,Central Station,-33.884024,151.206203,1.0,<NA>,0.0,<NA>,<NA>,POINT (151.2062 -33.88402)
2,201510,NaN,Redfern Station,-33.892048,151.198419,1.0,<NA>,0.0,<NA>,<NA>,POINT (151.19842 -33.89205)
3,204210,NaN,Newtown Station,-33.897913,151.179377,1.0,<NA>,0.0,<NA>,<NA>,POINT (151.17938 -33.89791)
4,204420,NaN,Sydenham Station,-33.914459,151.166526,1.0,<NA>,0.0,<NA>,<NA>,POINT (151.16653 -33.91446)


In [84]:
site_points = site_features.copy()
site_points["geometry"] = site_points.geometry.representative_point()
site_points = gpd.GeoDataFrame(site_points, geometry="geometry", crs=site_features.crs)

print("Site points CRS:", site_points.crs)
site_points.head()

Site points CRS: EPSG:4326


,RID,gurasid,propertytype,valnetpropertystatus,valnetpropertytype,dissolveparcelcount,valnetlotcount,propid,superlot,address,...,primary_bushfire_category,bushfire_risk_level,has_bushfire_cat1,has_bushfire_buffer,flood_flag,flood_hit_count,flood_class_count,flood_classes,flood_comments,primary_flood_class
0,260,49494103,1,2.0,2.0,1,1,1341,N,44 NORTHCOTE STREET ABERDARE,...,None,0,0,0,0,0,0,,,None
1,319,49494159,1,2.0,2.0,1,1,3169,N,10 MEREWETHER CLOSE NORTH ROTHBURY,...,Vegetation Category 1,3,1,1,0,0,0,,,None
2,327,49494166,1,2.0,2.0,1,1,3682,N,112 MATHIESON STREET BELLBIRD HEIGHTS,...,None,0,0,0,0,0,0,,,None
3,338,49494176,1,2.0,2.0,1,1,3958,N,584 WOLLOMBI ROAD BELLBIRD,...,Vegetation Buffer,1,0,1,0,0,0,,,None
4,359,49494197,1,2.0,2.0,1,1,4739,N,41 RUSSELL STREET BRANXTON,...,None,0,0,0,0,0,0,,,None


In [85]:
TARGET_CRS = "EPSG:3857"

site_points_m = site_points.to_crs(TARGET_CRS)
station_points_m = station_points.to_crs(TARGET_CRS)

print(site_points_m.crs)
print(station_points_m.crs)

EPSG:3857
EPSG:3857


In [86]:
station_keep_cols = [
    c for c in ["stop_id", "stop_name", "route_type", "location_type", "geometry"]
    if c in station_points_m.columns
]
station_points_m = station_points_m[station_keep_cols].copy()

station_points_m.head()

,stop_id,stop_name,location_type,geometry
0,2000171,QVB,1.0,POINT (16832247.898 -4011586.423)
1,200060,Central Station,1.0,POINT (16832197.532 -4013239.926)
2,201510,Redfern Station,1.0,POINT (16831331.019 -4014315.863)
3,204210,Newtown Station,1.0,POINT (16829211.307 -4015102.461)
4,204420,Sydenham Station,1.0,POINT (16827780.748 -4017321.73)


In [87]:
nearest_station = gpd.sjoin_nearest(
    site_points_m,
    station_points_m,
    how="left",
    distance_col="distance_to_station_m"
)

In [88]:
cols = ["RID", "address", "distance_to_station_m"]
for c in ["stop_name", "stop_id"]:
    if c in nearest_station.columns:
        cols.append(c)

print(len(nearest_station))
nearest_station[cols].head(10)

49950


,RID,address,distance_to_station_m,stop_name,stop_id
0,260,44 NORTHCOTE STREET ABERDARE,18148.199692,Lochinvar Station,232110
1,319,10 MEREWETHER CLOSE NORTH ROTHBURY,3392.611518,Branxton Station,233510
2,327,112 MATHIESON STREET BELLBIRD HEIGHTS,21134.025534,Lochinvar Station,232110
3,338,584 WOLLOMBI ROAD BELLBIRD,24462.595670,Lochinvar Station,232110
4,359,41 RUSSELL STREET BRANXTON,291.915278,Branxton Station,233510
5,400,14 FOSTER STREET CESSNOCK,17914.216915,Lochinvar Station,232110
6,456,13 WILLIAM STREET CESSNOCK,18877.238657,Lochinvar Station,232110
7,606,761 WATAGAN CREEK ROAD WATAGAN,34679.729206,Morisset Station,226410
8,913,4 CORAL CRESCENT PEARL BEACH,7618.186083,Woy Woy Station,225610
9,951,65 DONALD AVENUE UMINA BEACH,3139.912501,Woy Woy Station,225610


In [89]:
nearest_station["within_800m_catchment"] = (
    nearest_station["distance_to_station_m"] <= 800
).astype(int)

In [90]:
nearest_station["log_distance_to_station_m"] = np.log1p(nearest_station["distance_to_station_m"])
nearest_station["station_access_score"] = 1 / (1 + nearest_station["distance_to_station_m"])

In [91]:
print(nearest_station["distance_to_station_m"].describe())
print(nearest_station["within_800m_catchment"].value_counts(normalize=True, dropna=False))

count     49950.000000
mean      51000.380525
std      115602.613872
min          14.787625
25%         993.381745
50%        2650.741353
75%       25425.353470
max      991003.183556
Name: distance_to_station_m, dtype: float64
within_800m_catchment
0    0.794955
1    0.205045
Name: proportion, dtype: float64


In [92]:
view_cols = ["RID", "address", "distance_to_station_m", "within_800m_catchment"]
for c in ["stop_name", "stop_id"]:
    if c in nearest_station.columns:
        view_cols.append(c)

nearest_station[view_cols].sample(20, random_state=42)

,RID,address,distance_to_station_m,within_800m_catchment,stop_name,stop_id
8194,893252,225 WATKINS ROAD WANGI WANGI,9317.476976,0,Awaba Station,228310
34140,4368209,622/5 HUTCHINSON WALK ZETLAND,1033.711910,0,Green Square Station,201710
38516,5372408,18 MISTFUL PARK ROAD GOULBURN,4230.393235,0,Goulburn Station,258010
48605,7352126,7307/117 BATHURST STREET SYDNEY,165.385472,1,Town Hall Station,200070
35605,4735263,164A YORK ROAD SOUTH PENRITH,3935.669808,0,Penrith Station,275010
29229,3395584,26/553 NEW CANTERBURY ROAD DULWICH HILL,1823.330053,0,Lewisham Station,204920
4413,482527,4 MYALLIE AVENUE BAULKHAM HILLS,3974.959247,0,Toongabbie Station,214610
24674,2773457,2 GUERNSEY STREET SCONE,216.220834,1,Scone Station,233710
10783,1170814,11/1351 PITTWATER ROAD NARRABEEN,5904.552566,0,Pittwater Rd At Warringah Rd,G210019
21421,2349167,7 KALGOORLIE STREET WILLOUGHBY,1428.921945,0,Artarmon Station,206410


In [93]:
transport_features = nearest_station[
    [
        "RID",
        "distance_to_station_m",
        "within_800m_catchment",
        "log_distance_to_station_m",
        "station_access_score",
    ]
].copy()

transport_features.head()

,RID,distance_to_station_m,within_800m_catchment,log_distance_to_station_m,station_access_score
0,260,18148.199692,0,9.806382,0.000055
1,319,3392.611518,0,8.129650,0.000295
2,327,21134.025534,0,9.958687,0.000047
3,338,24462.595670,0,10.104941,0.000041
4,359,291.915278,1,5.679883,0.003414


In [94]:
site_with_transport = site_features.merge(
    transport_features,
    on="RID",
    how="left"
)

site_with_transport = gpd.GeoDataFrame(
    site_with_transport,
    geometry="geometry",
    crs=site_features.crs,
)

print(len(site_with_transport))
site_with_transport.head()

49950


,RID,gurasid,propertytype,valnetpropertystatus,valnetpropertytype,dissolveparcelcount,valnetlotcount,propid,superlot,address,...,flood_flag,flood_hit_count,flood_class_count,flood_classes,flood_comments,primary_flood_class,distance_to_station_m,within_800m_catchment,log_distance_to_station_m,station_access_score
0,260,49494103,1,2.0,2.0,1,1,1341,N,44 NORTHCOTE STREET ABERDARE,...,0,0,0,,,None,18148.199692,0,9.806382,0.000055
1,319,49494159,1,2.0,2.0,1,1,3169,N,10 MEREWETHER CLOSE NORTH ROTHBURY,...,0,0,0,,,None,3392.611518,0,8.129650,0.000295
2,327,49494166,1,2.0,2.0,1,1,3682,N,112 MATHIESON STREET BELLBIRD HEIGHTS,...,0,0,0,,,None,21134.025534,0,9.958687,0.000047
3,338,49494176,1,2.0,2.0,1,1,3958,N,584 WOLLOMBI ROAD BELLBIRD,...,0,0,0,,,None,24462.595670,0,10.104941,0.000041
4,359,49494197,1,2.0,2.0,1,1,4739,N,41 RUSSELL STREET BRANXTON,...,0,0,0,,,None,291.915278,1,5.679883,0.003414


In [95]:
print(site_with_transport["distance_to_station_m"].describe())
print(site_with_transport["within_800m_catchment"].value_counts(normalize=True, dropna=False))

count     49950.000000
mean      51000.380525
std      115602.613872
min          14.787625
25%         993.381745
50%        2650.741353
75%       25425.353470
max      991003.183556
Name: distance_to_station_m, dtype: float64
within_800m_catchment
0    0.794955
1    0.205045
Name: proportion, dtype: float64


In [98]:
site_with_transport.sort_values("distance_to_station_m", ascending=True)[
    ["RID", "address", "distance_to_station_m", "within_800m_catchment"]
].head(20)

,RID,address,distance_to_station_m,within_800m_catchment
46273,6992966,6/205 MAROUBRA ROAD MAROUBRA,14.787625,1
22310,2448217,6 NORWOOD AVENUE GOONELLABAH,17.727628,1
20320,2229959,18/429-481 GEORGE STREET SYDNEY,20.279811,1
41085,5975540,500 CROWN STREET SURRY HILLS,21.165350,1
41086,5975564,500 CROWN STREET SURRY HILLS,21.165350,1
34045,4338677,GREAT WESTERN HIGHWAY HAZELBROOK,21.676979,1
7911,863644,20/145 NEW SOUTH HEAD ROAD VAUCLUSE,21.885619,1
40251,5811221,1503/226 VICTORIA STREET POTTS POINT,22.565903,1
40250,5811189,201/226 VICTORIA STREET POTTS POINT,22.565903,1
45248,6807117,4/206 VICTORIA ROAD BELLEVUE HILL,23.902681,1


In [99]:
site_with_transport.sort_values("distance_to_station_m", ascending=False)[
    ["RID", "address", "distance_to_station_m", "within_800m_catchment"]
].head(20)

,RID,address,distance_to_station_m,within_800m_catchment
26844,3059811,36 STURT STREET TIBOOBURRA,991003.183556,0
26845,3059822,16 STURT STREET TIBOOBURRA,990913.326519,0
48105,7285606,1598 MOUNT WOOWOOLAHRA ROAD BROUGHAMS GATE,963419.894261,0
47602,7205809,954 WILANGEE ROAD SILVERTON,954992.052008,0
26284,2979049,CANOPUS STREET SILVERTON,953660.213584,0
26281,2978742,GIPPS STREET SILVERTON,953053.893362,0
26009,2941081,STURT STREET SILVERTON,953034.885512,0
26008,2941079,BURKE STREET SILVERTON,952845.816866,0
24854,2794447,STIRLING STREET SILVERTON,952564.500833,0
25899,2924959,SILVERTON,951658.025077,0


In [101]:
site_with_transport["station_distance_bucket"] = pd.cut(
    site_with_transport["distance_to_station_m"],
    bins=[-1, 800, 2000, 5000, 10000, np.inf],
    labels=["<=800m", "800m-2km", "2km-5km", "5km-10km", ">10km"]
)

site_with_transport["station_distance_bucket"].value_counts(dropna=False)

station_distance_bucket
>10km       14495
800m-2km    10978
2km-5km     10817
<=800m      10242
5km-10km     3418
Name: count, dtype: int64

In [103]:
Path("../../data/processed/site_features").mkdir(parents=True, exist_ok=True)

site_with_transport.to_parquet(
    "../../data/processed/site_features/property_site_features_v1_sample.parquet",
    index=False
)

In [104]:
station_points = stations_gdf.copy()
print("Station points kept:", len(station_points))

Station points kept: 1284


In [105]:
print(site_with_transport["distance_to_station_m"].describe())
print(site_with_transport["within_800m_catchment"].value_counts(normalize=True, dropna=False))

count     49950.000000
mean      51000.380525
std      115602.613872
min          14.787625
25%         993.381745
50%        2650.741353
75%       25425.353470
max      991003.183556
Name: distance_to_station_m, dtype: float64
within_800m_catchment
0    0.794955
1    0.205045
Name: proportion, dtype: float64
